# Two-Stage Pipeline: General + Bridge
General 3-class model → Bridge model (overskriver, unntatt bygninger).

In [ ]:
%load_ext autoreload
%autoreload 2
import os, sys
from pathlib import Path
import hydra, numpy as np, pandas as pd, torch

PROJECT_ROOT = Path('/cluster/home/larshfle/superpoint_transformer_new')
sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)

from src.utils import init_config
from src.transforms import *
from src.data import *
from scripts.inference_pipeline import (
    load_model, run_inference, compute_scores,
    OUT_GROUND, OUT_NOT_GROUND, OUT_BUILDING, OUT_BRIDGE,
    OUT_CLASS_NAMES, OUT_CLASS_COLORS,
)
print('OK')


## 1. Settings

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'

# General 3-class model
GENERAL_CKPT = str(PROJECT_ROOT / 'logs/train/runs/2026-05-16_09-58-36/checkpoints/epoch_399.ckpt')
GENERAL_DATA = '/cluster/home/larshfle/datasets/norway_combined_3class'

# Bridge model
BRIDGE_CKPT  = str(PROJECT_ROOT / 'logs/train/runs/2026-05-06_19-01-24/checkpoints/epoch_399.ckpt')
BRIDGE_DATA  = '/cluster/home/larshfle/datasets/bro'

TILE          = 'oslo_32-1-515-133-77'
SPLIT         = 'test'
BUILDING_MASK = True   # False = la bro overskrive bygning
MAX_POINTS    = 150_000


## 2. Last modeller

In [ ]:
print("Laster general modell...")
cfg_gen, model_gen = load_model("semantic/norway_combined_3class", GENERAL_CKPT, device)
cfg_gen.datamodule.data_dir = GENERAL_DATA
dm_gen = hydra.utils.instantiate(cfg_gen.datamodule)
dm_gen.prepare_data(); dm_gen.setup()
dataset_gen = getattr(dm_gen, f"{SPLIT}_dataset")

print("Laster bro-modell...")
cfg_bro, model_bro = load_model("semantic/bro", BRIDGE_CKPT, device)
cfg_bro.datamodule.data_dir = BRIDGE_DATA
dm_bro = hydra.utils.instantiate(cfg_bro.datamodule)
dm_bro.prepare_data(); dm_bro.setup()
dataset_bro = getattr(dm_bro, f"{SPLIT}_dataset")
print('Modeller lastet!')


## 3. Inferens

In [ ]:
print("Stage 1: General modell...")
pos, general_pred = run_inference(dataset_gen, TILE, model_gen, device)
print(f"  {len(pos):,} punkter")

print("Stage 2: Bro-modell...")
_, bridge_pred = run_inference(dataset_bro, TILE, model_bro, device)
print(f"  {(bridge_pred==1).sum().item():,} bro-prediksjoner")


## 4. Merge prediksjoner

In [ ]:
gen_map = {0: OUT_GROUND, 1: OUT_NOT_GROUND, 2: OUT_BUILDING}
final = torch.full((len(pos),), 255, dtype=torch.long)
for src, dst in gen_map.items():
    final[general_pred == src] = dst

is_bridge   = bridge_pred == 1
is_building = general_pred == 2
mask = is_bridge & ~is_building if BUILDING_MASK else is_bridge
final[mask] = OUT_BRIDGE

print("Klassefordeling etter merge:")
for cid, name in enumerate(OUT_CLASS_NAMES):
    n = (final == cid).sum().item()
    print(f"  {name:<12}: {n:>10,}  ({100*n/len(final):.2f}%)")
if BUILDING_MASK:
    blocked = (is_bridge & is_building).sum().item()
    print(f"  Blokkert av bygningsmaske: {blocked:,}")


## 5. Visualisering

In [ ]:
import plotly.graph_objects as go

# Bygg ett NAG fra general-modellen for visualisering
subtile_ids = [i for i, cid in enumerate(dataset_gen.cloud_ids) if cid.startswith(TILE)]
nag_vis = dataset_gen[subtile_ids[0]]
nag_vis = dataset_gen.on_device_transform(nag_vis.to(device))
with torch.no_grad():
    out_vis = model_gen(nag_vis)
nag_vis[0].semantic_pred = out_vis.voxel_semantic_pred(super_index=nag_vis[0].super_index)

# Overskriver predictions med bridge
n_vox = nag_vis[0].semantic_pred.shape[0]
nag_vis[0].semantic_pred[:n_vox] = nag_vis[0].semantic_pred.clone()

nag_vis.show(
    class_names=OUT_CLASS_NAMES + ["ignored"],
    class_colors=OUT_CLASS_COLORS + [[50,50,50]],
    stuff_classes=list(range(4)),
    num_classes=4,
    max_points=MAX_POINTS,
    semantic_pred=True,
)


## 6. Eksporter LAS

In [ ]:
import laspy

pos_np   = pos.numpy().astype(np.float64)
final_np = final.numpy().astype(np.uint8)

colors   = np.array(OUT_CLASS_COLORS + [[50,50,50]], dtype=np.uint16)
rgb      = (colors[np.clip(final_np, 0, len(colors)-1)] * 256).astype(np.uint16)

header = laspy.LasHeader(point_format=2, version="1.2")
las_out = laspy.LasData(header=header)
las_out.x = pos_np[:,0]; las_out.y = pos_np[:,1]; las_out.z = pos_np[:,2]
las_out.red = rgb[:,0]; las_out.green = rgb[:,1]; las_out.blue = rgb[:,2]
las_out.classification = final_np

out_path = PROJECT_ROOT / f'output_pipeline/{TILE}_pipeline.las'
out_path.parent.mkdir(exist_ok=True)
las_out.write(str(out_path))
print(f"Lagret: {out_path}")
